<a href="https://colab.research.google.com/github/harpuneet-k/Celebal-Assignments/blob/main/Assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT 6**


Train multiple machine learning models and evaluate their performance using metrics such as accuracy, precision, recall, and F1-score.

Implement hyperparameter tuning techniques like GridSearchCV and RandomizedSearchCV to optimize model parameters.

Analyze the results to select the best-performing model.

In [1]:
# Step 1: Import Required Libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Step 2: Load and Prepare the Data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [3]:
# Step 3: Initialize Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}


In [4]:
# Step 4: Train and Evaluate Each Model
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })

# Display results
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("Model Performance Before Tuning:")
print(results_df)


Model Performance Before Tuning:
                 Model  Accuracy  Precision    Recall  F1-Score
2                  SVM  0.982456   0.972603  1.000000  0.986111
0  Logistic Regression  0.973684   0.972222  0.985915  0.979021
1        Random Forest  0.964912   0.958904  0.985915  0.972222
3                  KNN  0.947368   0.957746  0.957746  0.957746


In [5]:
# Step 5: Hyperparameter Tuning (Example for Random Forest)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(RandomForestClassifier(), param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)

print("\nBest Parameters for Random Forest using GridSearchCV:")
print(grid_rf.best_params_)



Best Parameters for Random Forest using GridSearchCV:
{'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}


In [6]:
# Step 6: Hyperparameter Tuning (Example for SVM using RandomizedSearchCV)
param_dist_svm = {
    'C': np.logspace(-2, 2, 10),
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf', 'poly']
}

random_search_svm = RandomizedSearchCV(SVC(), param_distributions=param_dist_svm, n_iter=10, cv=5, scoring='f1', random_state=42, n_jobs=-1)
random_search_svm.fit(X_train_scaled, y_train)

print("\nBest Parameters for SVM using RandomizedSearchCV:")
print(random_search_svm.best_params_)



Best Parameters for SVM using RandomizedSearchCV:
{'kernel': 'linear', 'gamma': 'scale', 'C': np.float64(0.0774263682681127)}


In [7]:
# Step 7: Retrain the Best Tuned Models and Evaluate
best_rf = grid_rf.best_estimator_
best_svm = random_search_svm.best_estimator_

final_models = {
    "Tuned Random Forest": best_rf,
    "Tuned SVM": best_svm
}

print("\nFinal Evaluation after Hyperparameter Tuning:")

for name, model in final_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"\n{name}:")
    print(classification_report(y_test, y_pred))



Final Evaluation after Hyperparameter Tuning:

Tuned Random Forest:
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Tuned SVM:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        43
           1       0.97      1.00      0.99        71

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [8]:
# Step 8: Summarize All Final Results in a Table
summary = []

for name, model in final_models.items():
    y_pred = model.predict(X_test_scaled)
    summary.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

summary_df = pd.DataFrame(summary).sort_values(by="F1-Score", ascending=False)
print("\nSummary Table (After Tuning):")
print(summary_df)



Summary Table (After Tuning):
                 Model  Accuracy  Precision    Recall  F1-Score
1            Tuned SVM  0.982456   0.972603  1.000000  0.986111
0  Tuned Random Forest  0.964912   0.958904  0.985915  0.972222
